# BNP Paribas- Bussines Case Project 2025/2026
**DATA VISUALIZATION AND PREPROCESSING NOTEBOOK**

**Group V**:
   - Alano Gonçalves (20250457)
   - Catarina Martins (20221914)
   - João Carichas (20250507)
   - Marta Ribeiro (20221886)
   - Nicole Nogueira(20221961)

<div class="alert alert-block alert-info">

<a class="anchor" id="1. Import">    </a>
# 1. Import
       
</div>


<a class="anchor" id="1.1 Import Libraries">

## 1.1 Import Libraries
    
</a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [2]:
# Configurações de estilo para os gráficos da apresentação
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

<a class="anchor" id="1.1 Import Libraries">

## 1.2 Data Loading
    
</a>

In [3]:
#file_path_BNP = r"C:\Users\alano\Desktop\Mestrado\2- Semestre\Busines case\BNP\Projeto GIt\crc.parquet"
file_path_BNP_teste = r"C:\Users\Catarina\Documents\GitHub\BNP-Paribas\Preprocessed_dataset_BNP2_teste.csv"

# Ler o ficheiro parquet
#BNP = pd.read_csv(file_path_BNP)
BNP_teste = pd.read_csv(file_path_BNP_teste)

# Carregar o dataset pré-processado
df = pd.read_csv(file_path_BNP_teste)

In [4]:
# Converter colunas temporais
date_cols = ["DCREAT", "CLOSE_DATE", "NEXT_DOSSIER_DATE"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.")

Dataset carregado: 185600 linhas e 55 colunas.


<a class="anchor" id="1.1 Import Libraries">

## 2. Target Engineering
    
</a>

In [5]:
# 1. Converter o Target para Binário (0 e 1)
# Usamos .fillna(0) porque os contratos abertos ainda não têm churn definido
df['TARGET'] = df['CHURN'].fillna(0).astype(int)

# 2. SELEÇÃO ANTI-LEAKAGE E REMOÇÃO DE IDs
# incluir aqui todas as colunas que identificam o cliente ou o contrato
leaky_features = [
    'CHURN', 'NEXT_DOSSIER_DATE', 'DAYS_TO_NEXT', 
    'CHURN_DATE', 'days_to_churn', 'event', 'duration'
]

# Identificadores que não devem ir para o treino
ids_to_drop = ['CONTRIB', 'DCREAT', 'CLOSE_DATE', 'OBS_END_DATE']

# Criamos X (features) e y (alvo)
# O código verifica se a coluna existe antes de tentar fazer o drop para evitar erros
cols_to_drop = [col for col in leaky_features + ids_to_drop if col in df.columns]
X = df.drop(columns=cols_to_drop + ['TARGET'])
y = df['TARGET']

print(f"Colunas removidas: {cols_to_drop}")

Colunas removidas: ['CHURN', 'NEXT_DOSSIER_DATE', 'DAYS_TO_NEXT', 'CONTRIB', 'DCREAT', 'CLOSE_DATE', 'OBS_END_DATE']


## Encoding

In [6]:
# 1. Identificar colunas categóricas
cat_features = X.select_dtypes(include=['object']).columns

for col in cat_features:
    n_categories = X[col].nunique()
    
    # A. One-Hot Encoding para categorias binárias ou muito pequenas (<= 3 categorias)
    if n_categories <= 3:
        X = pd.get_dummies(X, columns=[col], prefix=col, drop_first=True)
        print(f"One-Hot Encoding aplicado a: {col}")
        
    # B. Frequency Encoding para o resto
    else:
        freq = X[col].value_counts(normalize=True)
        X[col + '_freq'] = X[col].map(freq)
        X.drop(columns=[col], inplace=True)
        print(f"Frequency Encoding aplicado a: {col} ({n_categories} categorias)")

# Garantir que não ficaram nulos após o encoding
X = X.fillna(0)

Frequency Encoding aplicado a: DOSSIER (185592 categorias)
Frequency Encoding aplicado a: DCRD_0 (22 categorias)
One-Hot Encoding aplicado a: POLE
Frequency Encoding aplicado a: PRODALP (4 categorias)
One-Hot Encoding aplicado a: PAGAMENTO
Frequency Encoding aplicado a: NATIO (4 categorias)
Frequency Encoding aplicado a: MODCONTACTO (6 categorias)
Frequency Encoding aplicado a: kp_sqe (16 categorias)
Frequency Encoding aplicado a: sdem_SITFAM (7 categorias)
Frequency Encoding aplicado a: sdem_HABITAT (8 categorias)


In [7]:
X.head()

,is_san,is_sol,DURDEG,RANGPRO_max,RANGCLI_max,delay_sum,MENSALIDADE,CRD_min,MTFINO,RESSO_mean,...,DCRD_0_freq,POLE_P,PRODALP_freq,PAGAMENTO_N,PAGAMENTO_P,NATIO_freq,MODCONTACTO_freq,kp_sqe_freq,sdem_SITFAM_freq,sdem_HABITAT_freq
0,1,0,84,3,19,1,158.359889,0.000,8000.00,1395.9935,...,0.064663,True,0.743233,False,True,0.989628,0.607220,0.032015,0.186498,0.277527
1,1,0,120,13,19,1,280.401466,0.000,16000.00,1513.4660,...,0.033836,True,0.743233,False,True,0.989628,0.607220,0.032015,0.186498,0.277527
2,0,0,120,91,91,0,347.447280,8115.247,20000.00,1113.2580,...,0.000000,True,0.743233,False,True,0.989628,0.384892,0.032678,0.400921,0.271105
3,0,1,60,21,21,0,424.699203,0.000,13467.54,678.4830,...,0.086581,True,0.743233,False,True,0.989628,0.006320,0.694133,0.400921,0.350770
4,0,1,60,21,21,0,184.214593,0.000,5985.57,678.4830,...,0.086581,True,0.743233,False,True,0.989628,0.006320,0.694133,0.400921,0.350770


<a class="anchor" id="1.1 Import Libraries">

## 3. Time-Based Split
    
</a>

In [8]:
# 1. Separar o que é PASSADO do que é o FUTURO (Predição Real)
# Histórico: Contratos terminados (onde sabemos se houve churn ou não)
historical_data = df[df['CLOSE_DATE'].notna()].sort_values('CLOSE_DATE')

# Predição Real: Contratos ainda abertos (o nosso verdadeiro Teste)
test_data = df[df['CLOSE_DATE'].isna()]

# 2. Definir o Ponto de Corte para a Validação (Time-based)
# Vamos usar os últimos 4 meses de histórico para validar a performance
cutoff_date = historical_data['CLOSE_DATE'].max() - pd.DateOffset(months=4)

train_set = historical_data[historical_data['CLOSE_DATE'] < cutoff_date]
val_set = historical_data[historical_data['CLOSE_DATE'] >= cutoff_date]

# 3. Preparar as Matrizes (X) e o Alvo (y)
# Treino (Onde o modelo aprende)
X_train = train_set.drop(columns=['TARGET'])
y_train = train_set['TARGET']

# Validação (Onde medimos a precisão antes de apresentar ao CEO)
X_val = val_set.drop(columns=['TARGET'])
y_val = val_set['TARGET']

# Teste Real (Onde vamos aplicar o modelo para prever o risco atual)
X_test = test_data.drop(columns=['TARGET'])

print(f"Dataset Total: {len(df)} linhas")
print(f"--- Treino (Passado Remoto): {len(X_train)} dossiers")
print(f"--- Validação (Passado Recente): {len(X_val)} dossiers")
print(f"--- Teste Real (Dossiers em Curso): {len(X_test)} dossiers")

Dataset Total: 185600 linhas
--- Treino (Passado Remoto): 63118 dossiers
--- Validação (Passado Recente): 13528 dossiers
--- Teste Real (Dossiers em Curso): 108954 dossiers


<a class="anchor" id="1.1 Import Libraries">

## 4. Feature Selection
    
</a>

### 4.1 Filter Methods

In [ ]:
#checking the variance of X_train
X_val.var()

In [ ]:
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]